# Capítulo 2 · Populações, amostras e dados: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa os cálculos do caso e confere os números. Todos os dados são **sintéticos** e os valores têm apenas finalidade didática.

A **Parte A** percorre o caso do capítulo, a base de altas para casa de janeiro de um hospital de 500 leitos, pelas perguntas do gestor; aparecem só as seções com cálculo (2.7, 2.9 e 2.10). A **Parte B** traz o código e o resultado de cada exercício do capítulo que precisa de código (7, 8 e 9; os enunciados estão no livro). O código é Python 3 simples, só com a biblioteca padrão. Execute as células em ordem.

## Parte A. O caso

**Dados.** `data/register.csv`: 1.200 linhas, uma por episódio de alta de janeiro de 2026, e cinco colunas: `episode_id`, `patient_id`, `formalized_at` (quando a alta foi formalizada), `departed_at` (quando o paciente saiu) e `delayed` (já calculada a partir dos dois horários; a primeira célula a calcula de novo). Cada célula traz o número da seção do livro a que pertence.

### 2.7 Definindo o evento a ser contado

*Quando uma alta conta como atrasada, e quantos dos 120 episódios sorteados foram?*

In [1]:
# PROGRAMA DE AUDITORIA: quanto esperaram os pacientes sorteados depois de a
# alta ser formalizada, e quantos esperaram mais de 2 horas?
#
# Pseudocódigo
#   1. Ler a base de altas de janeiro, um registro por episódio.
#   2. Para cada episódio, calcular a espera em horas, da alta formalizada
#      até a saída. Um horário faltante deixa a espera desconhecida.
#   3. Marcar o episódio como atrasado se a espera for MAIOR que 2 horas.
#   4. Sortear a amostra da auditoria: 120 episódios, ao acaso, sem reposição.
#   5. Relatar sobre a amostra: atrasados, porcentagem, mediana, maior, desconhecidos.
#   6. Contar de novo pela regra "2 horas ou mais", para ver o que
#      muda quando uma espera de exatamente 2 horas conta como atraso.

import csv
import random
import statistics
from datetime import datetime

TARGET_HOURS = 2  # a regra da auditoria: atrasado significa MAIS de 2 horas

# 1. Ler a base.
with open("data/register.csv", newline="") as stream:
    population = list(csv.DictReader(stream))

# 2-3. Calcular cada espera e marcar os atrasos.
for row in population:
    row["wait"] = None
    row["delayed"] = None
    if row["formalized_at"] and row["departed_at"]:
        formalized = datetime.fromisoformat(row["formalized_at"])
        departed = datetime.fromisoformat(row["departed_at"])
        wait = (departed - formalized).total_seconds() / 3600
        if wait >= 0:
            row["wait"] = wait
            row["delayed"] = wait > TARGET_HOURS
        else:
            print("Conferir horários:", row["episode_id"])

# 4. Sortear a amostra; a semente (11) torna o sorteio reproduzível.
sample = random.Random(11).sample(population, 120)

# 5. Relatar sobre a amostra.
waits = [row["wait"] for row in sample if row["wait"] is not None]
delayed = [wait for wait in waits if wait > TARGET_HOURS]
print("Episódios revisados:", len(sample))
print("Espera desconhecida:", len(sample) - len(waits))
print("Acima de 2 horas:", len(delayed), "de", len(waits))
print("Porcentagem acima de 2 horas:", 100 * len(delayed) / len(waits))
print("Espera mediana (horas):", statistics.median(waits))
print("Maior espera (horas):", max(waits))

# 6. As mesmas esperas pela regra "2 horas ou mais".
at_least_two = [wait for wait in waits if wait >= TARGET_HOURS]
share = 100 * len(at_least_two) / len(waits)
print("2 horas ou mais:", len(at_least_two), "de", len(waits))
print("Porcentagem de 2 horas ou mais:", share)

Episódios revisados: 120
Espera desconhecida: 0
Acima de 2 horas: 24 de 120
Porcentagem acima de 2 horas: 20.0
Espera mediana (horas): 1.5
Maior espera (horas): 8.0
2 horas ou mais: 45 de 120
Porcentagem de 2 horas ou mais: 37.5


### 2.9 Variação amostral

*Outro sorteio aleatório de 120 episódios daria a mesma proporção?*

In [2]:
# VARIAÇÃO AMOSTRAL: sorteios aleatórios diferentes dão a mesma proporção?
#
# Pseudocódigo
#   1. Contar os episódios atrasados na base inteira (só é possível
#      porque os dados são sintéticos; uma equipe de auditoria não a teria).
#   2. Sortear três amostras de 120 episódios, cada uma com uma semente diferente.
#   3. Em cada amostra, contar os episódios atrasados e a porcentagem.

def count_delayed(records):
    """Número de registros cuja situação de atraso é True (desconhecida não é contada)."""
    count = 0
    for row in records:
        if row["delayed"] is True:
            count = count + 1
    return count

# 1. A base inteira.
N = len(population)
K = count_delayed(population)
print(f"Base inteira: {K} atrasados de {N} ({100 * K / N:.2f}%)")

# 2-3. Três amostras; as sementes 11, 17 e 42 tornam os sorteios reproduzíveis.
sample_number = 0
for seed in [11, 17, 42]:
    sample_number = sample_number + 1
    draw = random.Random(seed).sample(population, 120)
    k = count_delayed(draw)
    n = len(draw)
    print(f"Amostra {sample_number}: {k} atrasados de {n} ({100 * k / n:.2f}%)")

Base inteira: 216 atrasados de 1200 (18.00%)
Amostra 1: 24 atrasados de 120 (20.00%)
Amostra 2: 22 atrasados de 120 (18.33%)
Amostra 3: 18 atrasados de 120 (15.00%)


### 2.10 Observações faltantes

*O que a equipe pode relatar quando falta o horário da alta formalizada em 12 dos 120 registros?*

In [3]:
# OBSERVAÇÕES FALTANTES: o que a equipe pode relatar quando faltam horários?
#
# Pseudocódigo
#   1. Fazer uma cópia separada dos 120 registros sorteados.
#   2. Na cópia, esconder a situação de atraso dos 12 primeiros episódios, como se
#      o horário da alta formalizada não tivesse sido registrado.
#   3. Contar as situações selecionadas, conhecidas e desconhecidas e os atrasos conhecidos.
#   4. Relatar a proporção de atrasos entre as situações conhecidas.
#   5. Relatar a faixa dos 120 episódios selecionados: toda situação desconhecida
#      no prazo (mínimo) ou toda situação desconhecida atrasada (máximo).

# 1-2. Copiar os registros (os originais ficam intactos) e esconder 12 situações.
incomplete = [dict(row) for row in sample]
for row in incomplete[:12]:
    row["delayed"] = None

# 3. Contagens.
selected = len(incomplete)
known = [row for row in incomplete if row["delayed"] is not None]
unknown = selected - len(known)
known_delays = count_delayed(known)

# 4-5. Resultados.
print("Selecionados:", selected)
print("Situação conhecida:", len(known))
print("Situação desconhecida:", unknown)
print("Atrasos conhecidos:", known_delays)
print(f"Atrasados entre os conhecidos: {100 * known_delays / len(known):.1f}%")
lowest = 100 * known_delays / selected
highest = 100 * (known_delays + unknown) / selected
print(f"Todos os selecionados: {lowest:.1f}% a {highest:.1f}%")

Selecionados: 120
Situação conhecida: 108
Situação desconhecida: 12
Atrasos conhecidos: 23
Atrasados entre os conhecidos: 21.3%
Todos os selecionados: 19.2% a 29.2%


## Parte B. Soluções dos exercícios

### 2.13 Exercício 7. Três sorteios de uma população-alvo

In [4]:
for seed in [11, 17, 42]:
    draw = random.Random(seed).sample(population, 120)
    identifiers = set()
    for row in draw:
        identifiers.add(row["episode_id"])
    assert len(draw) == len(identifiers) == 120
assert len(population) == 1200
assert K == 216

### 2.13 Exercício 8. Uma amostra incompleta

In [5]:
sample = random.Random(11).sample(population, 120)
incomplete = []
for row in sample:
    incomplete.append(dict(row))
for row in incomplete[:12]:
    row["delayed"] = None

known_n = 0
known_delays = 0
for row in incomplete:
    if row["delayed"] is not None:
        known_n = known_n + 1
        if row["delayed"] is True:
            known_delays = known_delays + 1
selected_n = len(incomplete)
missing_n = selected_n - known_n
print(selected_n, known_n, missing_n, known_delays)
# 120 108 12 23
print(f"Situações conhecidas: {100 * known_delays / known_n:.1f}%")
# Situações conhecidas: 21.3%
lower = 100 * known_delays / selected_n
upper = 100 * (known_delays + missing_n) / selected_n
print(f"Grupo selecionado: {lower:.1f}%-{upper:.1f}%")
# Grupo selecionado: 19.2%-29.2%

120 108 12 23
Situações conhecidas: 21.3%
Grupo selecionado: 19.2%-29.2%


### 2.13 Exercício 9. Definir a próxima auditoria de processo

In [6]:
# 1. Os seis eventos de um episódio sintético de desocupação de leito (da Figura 2.2)
events = [
    ("leito desocupado", "2026-01-04T13:00"),
    ("limpeza solicitada", "2026-01-04T13:20"),
    ("limpeza iniciada", "2026-01-04T14:10"),
    ("limpeza concluída", "2026-01-04T14:40"),
    ("leito pronto", "2026-01-04T14:50"),
    ("próximo paciente entra", "2026-01-04T15:30"),
]

# 2. Minutos entre cada evento e o seguinte, com o maior intervalo
total = 0
longest_name = ""
longest_minutes = 0
for i in range(len(events) - 1):
    name_a, time_a = events[i]
    name_b, time_b = events[i + 1]
    minutes = (datetime.fromisoformat(time_b) - datetime.fromisoformat(time_a)).total_seconds() / 60
    print(name_a, "até", name_b + ":", minutes, "minutos")
    total = total + minutes
    if minutes > longest_minutes:
        longest_minutes = minutes
        longest_name = name_a + " até " + name_b
print("Total da desocupação do leito até a entrada do próximo paciente:", total, "minutos")
print("Maior intervalo:", longest_name, "(" + str(longest_minutes) + " minutos)")

leito desocupado até limpeza solicitada: 20.0 minutos
limpeza solicitada até limpeza iniciada: 50.0 minutos
limpeza iniciada até limpeza concluída: 30.0 minutos
limpeza concluída até leito pronto: 10.0 minutos
leito pronto até próximo paciente entra: 40.0 minutos
Total da desocupação do leito até a entrada do próximo paciente: 150.0 minutos
Maior intervalo: limpeza solicitada até limpeza iniciada (50.0 minutos)


---
*Dados sintéticos (`generate_data.py`, sem números aleatórios). Valores apenas didáticos; nada aqui é estimativa para um hospital real.*